# COPIL Presentation Generator
Generateur de presentation PowerPoint depuis Google Drive


In [ ]:
!pip install -q python-pptx
from google.colab import drive
drive.mount('/content/drive')
print('OK - Google Drive monte')

In [ ]:
import os
import csv
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.enum.chart import XL_CHART_TYPE

DRIVE_FOLDER = '/content/drive/MyDrive/PF_SCORING'

if os.path.exists(DRIVE_FOLDER):
    files = [f for f in os.listdir(DRIVE_FOLDER) if f.lower().endswith('.csv')]
    if files:
        CSV_FILE = os.path.join(DRIVE_FOLDER, files[0])
        print(f'CSV trouve: {files[0]}')
    else:
        print('Aucun CSV dans le dossier')
else:
    print(f'Dossier non trouve: {DRIVE_FOLDER}')

In [ ]:
PRIMARY_BLUE = RGBColor(0, 51, 102)
ACCENT_ORANGE = RGBColor(255, 153, 0)
WHITE = RGBColor(255, 255, 255)
GREEN = RGBColor(76, 175, 80)
ORANGE = RGBColor(255, 152, 0)
RED = RGBColor(244, 67, 54)
DARK_TEXT = RGBColor(33, 33, 33)
LIGHT_GRAY = RGBColor(240, 240, 240)

def parse_completion(item):
    try:
        comp_str = item.get('COMPLETION_%', '0').rstrip('%').strip()
        return float(comp_str) if comp_str else 0
    except:
        return 0

def load_csv(csv_file):
    items = []
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        items = list(reader)
    print(f'Charges: {len(items)} elements')
    return items

def calc_stats(items):
    completions = [parse_completion(item) for item in items]
    stats = {
        'total': len(items),
        'integrated': len([i for i in items if i.get('STATUT') == 'INTEGRE']),
        'in_progress': len([i for i in items if i.get('STATUT') == 'EN COURS']),
        'planned': len([i for i in items if i.get('STATUT') == 'PLANIFIE']),
        'blocked': len([i for i in items if i.get('STATUT') == 'BLOQUE']),
        'by_bloc': {},
        'risks': [i for i in items if i.get('STATUT') == 'BLOQUE' or parse_completion(i) < 50]
    }
    stats['average_completion'] = sum(completions) / len(completions) if completions else 0
    for item in items:
        bloc = item.get('BLOC', 'Unknown')
        if bloc not in stats['by_bloc']:
            stats['by_bloc'][bloc] = {'total': 0, 'completion_sum': 0}
        stats['by_bloc'][bloc]['total'] += 1
        stats['by_bloc'][bloc]['completion_sum'] += parse_completion(item)
    for bloc in stats['by_bloc']:
        items_count = stats['by_bloc'][bloc]['total']
        stats['by_bloc'][bloc]['completion'] = stats['by_bloc'][bloc]['completion_sum'] / items_count if items_count > 0 else 0
    return stats

def create_ppt(stats, output_file):
    prs = Presentation()
    prs.slide_width = Inches(10)
    prs.slide_height = Inches(7.5)
    
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.background.fill
    bg.solid()
    bg.fore_color.rgb = PRIMARY_BLUE
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(2.5), Inches(9), Inches(1.5))
    p = title_box.text_frame.paragraphs[0]
    p.text = 'PF SCORING'
    p.font.size = Pt(66)
    p.font.bold = True
    p.font.color.rgb = WHITE
    p.alignment = PP_ALIGN.CENTER
    
    prs.save(output_file)
    print(f'PPT sauvegarde: {os.path.basename(output_file)}')

print('Fonctions pretes')

In [ ]:
if os.path.exists(CSV_FILE):
    items = load_csv(CSV_FILE)
    stats = calc_stats(items)
    output_file = os.path.join(DRIVE_FOLDER, 'COPIL_Presentation.pptx')
    create_ppt(stats, output_file)
    print('Succes! Verifiez votre Google Drive')
else:
    print('Erreur: CSV non trouve')